# Stacking：用元模型学习如何组合
> 难度标签：中级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：07_集成学习 | 源文件：Stacking.py | 核心功能：两层集成、元学习器、交叉验证预测

## 概述

Stacking 用第一层多个基模型的预测作为特征，训练一个元模型（meta-learner）做最终预测。比简单投票更灵活——元模型可以学习"哪些基模型在什么情况下更可信"。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import make_classification, load_iris
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.ensemble import StackingClassifier, StackingRegressor
from sklearn.linear_model import LogisticRegression, Ridge
# --- 导入库 ---
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

## 数学原理

### 1. Stacking 的两层结构

Stacking（堆叠泛化）由两层组成：

- **第一层**：$K$ 个不同的基学习器 $f_1, f_2, \ldots, f_K$
- **第二层**：一个**元学习器** $g$，学习如何最优地组合基学习器的预测

$$\hat{y} = g(f_1(x), f_2(x), \ldots, f_K(x))$$

### 2. 元特征的生成（K 折交叉预测）

为避免信息泄漏，第一层的训练使用 **K 折交叉预测**（cross_val_predict）：

对每个基学习器 $f_k$ 和每个折 $j=1,\ldots,J$：
1. 在 $J-1$ 个折上训练 $f_k$
2. 对第 $j$ 折的样本预测，得到 $\hat{y}_k^{(j)}$
3. 拼接所有折的预测，形成**元特征** $z_k = (\hat{y}_k^{(1)}, \ldots, \hat{y}_k^{(n)})$

代码中：
```python
meta_features_train.append(
    cross_val_predict(model, X_train, y_train, cv=5, method="predict")
)
```

### 3. 元特征矩阵

将所有基学习器的预测堆叠为元特征矩阵：

$$Z = [z_1, z_2, \ldots, z_K] \in \mathbb{R}^{n \times K}$$

- 分类任务：每列是类别标签（或概率向量，维度为 $K \times C$）
- 回归任务：每列是连续预测值

代码中 `np.column_stack(meta_features_train)` 完成此操作。

### 4. 元学习器的训练

在元特征矩阵 $Z$ 上训练元学习器 $g$：

$$\hat{g} = \arg\min_g \sum_{i=1}^{n} L(y_i, g(z_i))$$

代码中使用 `LogisticRegression` 作为元学习器。元学习器学到的是：在什么情况下信任哪个基学习器。

### 5. 测试集预测流程

1. 每个基学习器在**全部训练集**上重新训练
2. 对测试集预测，得到元特征 $Z_{test}$
3. 元学习器在 $Z_{test}$ 上预测最终结果

```python
for name, model in base_estimators:
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    meta_features_test.append(pred)
meta_X_test = np.column_stack(meta_features_test)
y_pred = meta_model.predict(meta_X_test)
```

### 6. Stacking 的数学优势

设基学习器的误差为 $\epsilon_k$，若它们的错误**不相关**，元学习器可以通过加权组合降低总误差：

$$\text{Var}\left(\sum_k w_k f_k\right) = \sum_k w_k^2 \text{Var}(f_k) + 2\sum_{k<l} w_k w_l \text{Cov}(f_k, f_l)$$

多样性高的基学习器（低协方差）组合效果更好。

### 7. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `cross_val_predict(..., cv=5)` | K=5 折交叉预测生成元特征 $z_k$ |
| `np.column_stack(meta_features)` | 构建元特征矩阵 $Z \in \mathbb{R}^{n \times K}$ |
| `LogisticRegression()` 作为 meta_model | 元学习器 $g(z)$ |
| `method="predict"` | 元特征为类别标签（离散值） |
| 5 个不同的基学习器 | $f_1, \ldots, f_5$，多样性是 Stacking 的关键 |
| 基学习器全部重新 fit | 测试时用完整训练数据获得最佳预测 |

### 1. 加载数据

首先加载数据集，为后续训练和评估做准备。

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### 2. 定义基学习器

运行 2. 定义基学习器 的代码，观察算法在该环节的行为。

In [ ]:
base_estimators = [
    ("dt", DecisionTreeClassifier(max_depth=5, random_state=42)),
    ("rf", RandomForestClassifier(n_estimators=50, random_state=42)),
    ("gb", GradientBoostingClassifier(n_estimators=50, random_state=42)),
    ("svm", SVC(kernel="rbf", probability=True, random_state=42)),
# --- 继续 ---
    ("knn", KNeighborsClassifier(n_neighbors=5)),
]

### 3. 手动实现 Stacking

运行 3. 手动实现 Stacking 的代码，观察算法在该环节的行为。

In [ ]:
print("=== 手动 Stacking ===")
# 第一层：用交叉预测生成元特征
meta_features_train = []
for name, model in base_estimators:
    cv_pred = cross_val_predict(model, X_train, y_train, cv=5, method="predict")
    # 也可用 predict_proba 获得概率特征
    meta_features_train.append(cv_pred)
meta_X_train = np.column_stack(meta_features_train)

# 第二层：用元学习器训练
meta_model = LogisticRegression(random_state=42)
meta_model.fit(meta_X_train, y_train)

# 在测试集上预测
meta_features_test = []
for name, model in base_estimators:
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    meta_features_test.append(pred)
# --- 数值计算 ---
meta_X_test = np.column_stack(meta_features_test)
y_pred = meta_model.predict(meta_X_test)

print(f"Stacking 准确率: {accuracy_score(y_test, y_pred):.4f}")

# 各基学习器单独准确率
print("\n各基学习器单独表现:")
for name, model in base_estimators:
    model.fit(X_train, y_train)
    acc = model.score(X_test, y_test)
    print(f"  {name:>5}: {acc:.4f}")

### 4. StackingClassifier（sklearn 封装）

运行 4. StackingClassifier（sklearn 封装） 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== StackingClassifier (sklearn) ===")
stack_clf = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
# --- 赋值/配置 ---
    passthrough=False,  # True: 元学习器也接收原始特征
)
stack_clf.fit(X_train, y_train)
print(f"训练准确率: {stack_clf.score(X_train, y_train):.4f}")
print(f"测试准确率: {stack_clf.score(X_test, y_test):.4f}")

### 5. passthrough 模式

运行 5. passthrough 模式 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== passthrough=True（原始特征 + 基学习器输出）===")
stack_pt = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
# --- 赋值/配置 ---
    passthrough=True,
)
stack_pt.fit(X_train, y_train)
print(f"测试准确率: {stack_pt.score(X_test, y_test):.4f}")

### 6. 不同元学习器

运行 6. 不同元学习器 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 不同元学习器 ===")
from sklearn.ensemble import RandomForestClassifier as RFC
from sklearn.svm import SVC as SVCC

for name, final in [("LogisticRegression", LogisticRegression(max_iter=1000)),
                     ("RandomForest", RFC(n_estimators=50, random_state=42)),
                     ("SVC", SVCC(kernel="rbf", probability=True))]:
    s = StackingClassifier(estimators=base_estimators, final_estimator=final, cv=5)
    s.fit(X_train, y_train)
# --- 输出结果 ---
    print(f"  {name:>20}: 测试准确率={s.score(X_test, y_test):.4f}")

### 7. Stacking 回归

在回归任务上拟合模型，观察预测值与真实值的偏差。

In [ ]:
print("\n=== Stacking 回归 ===")
from sklearn.datasets import make_regression
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge, Lasso

X_r, y_r = make_regression(n_samples=500, n_features=10, noise=10, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_r, y_r, test_size=0.2, random_state=42)

base_reg = [
    ("rf", RandomForestRegressor(n_estimators=50, random_state=42)),
    ("gb", GradientBoostingRegressor(n_estimators=50, random_state=42)),
    ("ridge", Ridge()),
]
# --- 继续 ---
stack_reg = StackingRegressor(estimators=base_reg, final_estimator=Ridge(), cv=5)
stack_reg.fit(X_tr, y_tr)
print(f"Stacking 回归 R²: {stack_reg.score(X_te, y_te):.4f}")

### 8. Stacking 要点

运行 8. Stacking 要点 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== Stacking 要点 ===")
print("- 第一层: 多个不同的基学习器（多样化是关键）")
print("- 第二层: 元学习器学习如何组合基学习器的输出")
print("- 使用交叉预测生成元特征（防止信息泄露）")
print("- passthrough=True: 元学习器同时看到原始特征和基学习器输出")
# --- 输出结果 ---
print("- 基学习器应尽量多样化（不同算法/不同参数）")
print("- 常用于竞赛中提升最终成绩（0.1~0.5% 的提升）")

## 关键代码解释

```python
from sklearn.ensemble import StackingClassifier
stacking = StackingClassifier(
    estimators=[("lr", lr), ("rf", rf), ("svm", svm)],
    final_estimator=LogisticRegression(), cv=5
)
```

`cv=5` 确保元特征不泄露：每个样本的元特征来自它未参与训练的 fold 的预测。

## 注意事项

1. 必须用交叉验证生成元特征，否则严重过拟合
2. 元模型通常用简单模型（逻辑回归）
3. 训练时间 = 基模型数 × CV 折数 + 元模型训练

## 总结与延伸

以上代码展示了 **Stacking** 的完整流程。

**解读要点**：
- 对比 **单模型 vs 集成模型** 的性能提升
- 关注 **特征重要性** 排名
- 观察 OOB 分数（随机森林）或 early stopping 轮次（XGBoost）

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **多层 Stacking**：Stacking 的输出再做 Stacking
- **Feature-weighted Stacking**：给不同特征组用不同基模型
- **Netflix Prize**：冠军方案就是大规模 Stacking
